### ЗАДАЧА: Бронирование переговорок

Офис-менеджер получает пачку заявок на переговорки.
Нужно собрать систему, которая:
- принимает корректные бронирования,
- отбрасывает конфликтующие или неправильные заявки,
- хранит расписание по комнатам,
- позволяет быстро понять, какая переговорка загружена сильнее всего.

В некоторых заявках указана неизвестная комната,
в некоторых время перепутано,
а некоторые пересекаются с уже занятыми слотами.


In [ ]:
from dataclasses import dataclass
from typing import Optional

rooms = {'A-101', 'B-204', 'C-305'}
# rows: booking_id|room_id|owner|start_hour|end_hour
rows = [
    'BK-100|A-101|Alice|9|11',
    'BK-101|A-101|Bob|10|12',
    'BK-102|B-204|Kira|13|15',
    'BK-103|X-999|Oleg|11|12',
    'BK-104|C-305|Eva|16|15',
    'BK-105|B-204|Max|15|17',
]

class BookingError(Exception):
    pass

class RoomNotFoundError(BookingError):
    pass

class TimeRangeError(BookingError):
    pass

class TimeConflictError(BookingError):
    pass

@dataclass(order=True)
class Booking:
    start_hour: int
    end_hour: int
    booking_id: str
    room_id: str
    owner: str

class RoomSchedule:
    def __init__(self, room_id):
        self.room_id = room_id
        self.bookings: list[Booking] = []

    def can_add(self, booking: Booking):
        for existing_booking in self.bookings:
            if not (booking.end_hour <= existing_booking.start_hour or 
                    booking.start_hour >= existing_booking.end_hour):
                return False
        return True

    def add_booking(self, booking):
        if not self.can_add(booking):
            raise TimeConflictError('Время пересекается')
        self.bookings.append(booking)
        self.bookings.sort(key=lambda x: x.start_hour)

    def total_reserved_hours(self):
        total = 0
        for booking in self.bookings:
            total += booking.end_hour - booking.start_hour
        return total

class BookingService:
    def __init__(self, rooms):
        self.schedules: dict[str, RoomSchedule] = {
            room_id: RoomSchedule(room_id) for room_id in rooms
        }
        self.accepted = []
        self.errors = []

    def parse_booking(self, row):
        parts = row.split('|')
        if len(parts) != 5:
            raise BookingError("Неверный формат записи")
        booking_id, room_id, owner, start_raw, end_raw = parts
        start_hour = int(start_raw)
        end_hour = int(end_raw)
        if room_id not in self.schedules:
            raise RoomNotFoundError(f"Комната не найдена {room_id}")
        if start_hour >= end_hour:
            raise TimeRangeError(f"Некорректное время: {start_hour}-{end_hour}")
        return Booking(start_hour, end_hour, booking_id, room_id, owner)

    def submit(self, row):
        try:
            booking = self.parse_booking(row)
            self.schedules[booking.room_id].add_booking(booking)
            self.accepted.append(booking)
        except BookingError as e:
            self.errors.append((row, type(e).__name__, str(e)))

    def load(self, rows):
        for row in rows:
            self.submit(row)

    def busiest_room(self):
        room_id = max(self.schedules, key=lambda room: self.schedules[room].total_reserved_hours())
        total_hours = self.schedules[room_id].total_reserved_hours()
        return room_id, total_hours

    def find_booking(self, booking_id) -> Optional[Booking]:
        for schedule in self.schedules.values():
            for booking in schedule.bookings:
                if booking.booking_id == booking_id:
                    return booking
        return None

service = BookingService(rooms)
service.load(rows)

print("Принятые бронирования:")
for booking in service.accepted:
    print(f"  {booking.booking_id}: {booking.room_id} с {booking.start_hour} до {booking.end_hour}, заказчик: {booking.owner}")

print("\nОшибки:")
for row, error_type, message in service.errors:
    print(f"  Строка '{row}': {error_type} — {message}")

print("\nРасписание по комнатам:")
for room_id, schedule in service.schedules.items():
    print(f"  Комната {room_id}:")
    if schedule.bookings:
        for booking in schedule.bookings:
            print(f"    {booking.owner}: {booking.start_hour}-{booking.end_hour} ({booking.booking_id})")
    else:
        print("    Нет бронирований")

busiest_room, hours = service.busiest_room()
print(f"\nСамая загруженная комната: {busiest_room} ({hours} часа)")

found_booking = service.find_booking('BK-102')
if found_booking:
    print(f"\nБронирование BK-102 найдено: {found_booking.owner} в комнате {found_booking.room_id}, {found_booking.start_hour}-{found_booking.end_hour}")
else:
    print("\nБронирование BK-102 не найдено")
